# Stage2 / 00 — Prepare datasets + taxonomy variants + remap sanity

Runs once at the top of the Stage2 session. Confirms that the
1-class (`stage2_1c_vehicle_merged`) and 3-class (`stage2_3c_extended`) class
protocols produce non-empty datasets and that BDD categories land
in the expected rows. Also verifies that the Stage1 best-row
checkpoint exists on Drive so Stage2 warm-start / distill runs
won't break at load time.

Outputs: a short report JSON saved to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
!pip install -q yacs tqdm opencv-python-headless

In [ ]:
import json, torch
import torchvision.transforms as T
from lib.config import cfg
from lib.dataset import BddDataset
from lib.dataset.class_maps import build_id_dict, available_protocols
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive, find_raw_bdd_root,
    resolve_bdd_images_100k_dir, resolve_bdd_labels_100k_dir,
)

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)
print('Available protocols:', available_protocols())

In [ ]:
# ── Class protocol remap sanity check ──
def count_classes_in(cfg_proto: str):
    cfg.defrost()
    cfg.DATASET.ROOT = DATASET_ROOT
    cfg.DATASET.DATAROOT = BDD_IMAGES
    cfg.DATASET.LABELROOT = BDD_LABELS
    cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')
    cfg.DATASET.CLASS_PROTOCOL = cfg_proto
    cfg.freeze()
    ds = BddDataset(cfg, is_train=False, inputsize=640, transform=T.ToTensor())
    counts = {}
    for item in ds.db[:500]:       # 500-sample spot check
        for row in item['label']:
            c = int(row[0])
            counts[c] = counts.get(c, 0) + 1
    return len(ds), counts

report = {}
for proto in ('stage2_1c_vehicle_merged', 'stage2_3c_extended'):
    n_total, counts = count_classes_in(proto)
    report[proto] = {'total_samples': n_total, 'class_counts_in_first_500': counts}
    print(f'[{proto}] samples={n_total} class_counts(first 500)={counts}')

In [ ]:
# ── Stage1 best-row checkpoint presence check ──
stage1_ckpt = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'checkpoints',
                           'yolopv2_best_row', 'best.pth')
report['stage1_best_row_ckpt'] = {
    'path': stage1_ckpt,
    'exists': os.path.exists(stage1_ckpt),
    'size_mb': (os.path.getsize(stage1_ckpt) / 1024**2) if os.path.exists(stage1_ckpt) else None,
}
print('Stage1 best-row checkpoint:', report['stage1_best_row_ckpt'])

# ── Persist the report ──
out = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage2', 'metrics', 'taxonomy_report.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w') as f:
    json.dump(report, f, indent=2)
print('Saved ->', out)